In [ ]:
import asyncio
import pandas as pd
from vpei.utils.llm_requests_v3 import *
from vpei.utils.llm_utils import save_model_experimental_results_to_csv
from vpei.common_utils import extract_score, extract_string
from vpei.common_variables import *
from vpei.epistemic_consistency.experiment_utils import *
from vpei.epistemic_consistency.experiment_types import carry_out_comparative_experiment_with_ground_truth_and_multiple_choices
from vpei.epistemic_consistency.active_prompts import EXPERIMENTS

input_file = "./data/code_snippets_sample.csv"
df = pd.read_csv(input_file, index_col=0)
df.rename(columns={"code_tokens":"code_snippet"}, inplace=True) # to make clearer prompt templates
# convert values of "Accepted" in column verdict into "No Runtime Error"
df['verdict'] = df['verdict'].replace('Accepted', 'No Runtime Error')
df = df.drop_duplicates(subset=['problem_id', 'submission_id'])

df_correct = df[(df["verdict"]=="No Runtime Error")]
df_incorrect = df[(df["verdict"]=="Runtime Error")]
df

In [ ]:
experiment_name = "code"
number_of_choices = 5
system_prompt = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["system_prompt"]
user_prompt_template_repeated_block = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_block"]
user_prompt_template_repeated_attribution_block = EXPERIMENTS[experiment_name]["comparative_experiment_with_ground_truth_and_multiple_choices"]["user_prompt_template_repeated_attribution_block"]
print(system_prompt)
print("-------------------------------------------------------------------")
print(build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block))

In [ ]:
# model_name = "gpt-4o-mini"
# model_name = "gpt-5"
from os import name


model_name = "gpt-5-mini"
model_kwargs = adapt_model_kwargs_for_model(model_name, custom_model_kwargs={})

user_prompt_template = build_user_prompt_template_with_variable_repeats(number_of_choices, user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block)

names_dict = {
"name_1": "P.P.",
"name_2": "A.J.",
"name_3": "L.K.",
"name_4": "M.R.",
"name_5": "S.T.",
}
political_attitudes = {
"political_attitude_1": "conservative",
"political_attitude_2": "progressive",
"political_attitude_3": "left-leaning",
"political_attitude_4": "right-wing",
"political_attitude_5": "Republican",
}

stimuli_factors_into_user_prompt = {
"code_snippet_1": df_incorrect.iloc[0]['code_snippet'],
"code_snippet_2": df_incorrect.iloc[0]['code_snippet'],
"code_snippet_3": df_incorrect.iloc[0]['code_snippet'],
"code_snippet_4": df_correct.iloc[0]['code_snippet'],
"code_snippet_5": df_incorrect.iloc[0]['code_snippet'],
}

user_prompt = user_prompt_template.format(**names_dict,
                                                      **political_attitudes,
                                                      **stimuli_factors_into_user_prompt)


messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
make_llm_request(model_name, messages, **model_kwargs)

In [ ]:


models = ["gpt-5-mini"]


n = 1
custom_model_kwargs = {}
stimuli_factors = ["code_snippet"]
additional_variables_from_df_to_save = [] 
number_of_choices = 5
path_to_save_model_outputs = "./comparative_experiment_with_ground_truth_and_multiple_choices"
random_seed = 42


In [ ]:
payloads = await carry_out_comparative_experiment_with_ground_truth_and_multiple_choices(models=models, df_correct=df_correct, df_incorrect=df_incorrect, n=n, 
                                                                                         system_prompt=system_prompt, user_prompt_template_repeated_block=user_prompt_template_repeated_block, user_prompt_template_repeated_attribution_block=user_prompt_template_repeated_attribution_block,
                                                                                         stimuli_factors=stimuli_factors, additional_variables_from_df_to_save=additional_variables_from_df_to_save,
                                                                                         custom_model_kwargs=custom_model_kwargs, path_to_save_model_outputs=path_to_save_model_outputs,
                                                                                         random_seed=21, number_of_choices=number_of_choices)

df = pd.DataFrame(payloads)
df['model_response_pole'].value_counts()